### ENVIRONMENT SETUP & DEPENDENCY INSTALLATION

In [2]:
!pip install pyodbc pandas sqlalchemy

Defaulting to user installation because normal site-packages is not writeable


### AZURE SQL CONNECTION TEST

In [3]:
import pyodbc
import pandas as pd
import warnings
warnings.filterwarnings('ignore') 

conn_str = "Driver={ODBC Driver 18 for SQL Server};Server=tcp:nashvilleserver2210.database.windows.net,1433;Database=NashvilleHousingDB;Uid=sqladmin;Pwd=Sr123456789@;Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"

print("Connecting to Azure SQL...")

try:
    conn = pyodbc.connect(conn_str)
    print("Connection successful! 🎉\n")

    query = "SELECT TOP 5 PropertySplitAddress, PropertySplitCity FROM dbo.NashvilleHousing;"

    df = pd.read_sql(query, conn)

    print("Here is your data fetched straight from the cloud:")
    print(df)
    
except Exception as e:
    print("Uh oh, the connection failed. Here is the error:")
    print(e)

Connecting to Azure SQL...
Connection successful! 🎉

Here is your data fetched straight from the cloud:
  PropertySplitAddress PropertySplitCity
0   1808  FOX CHASE DR    GOODLETTSVILLE
1   1832  FOX CHASE DR    GOODLETTSVILLE
2   1864 FOX CHASE  DR    GOODLETTSVILLE
3   1853  FOX CHASE DR    GOODLETTSVILLE
4   1829  FOX CHASE DR    GOODLETTSVILLE


### GEOSPATIAL DEPENDENCY INSTALLATION

In [4]:
!pip install geopy

Defaulting to user installation because normal site-packages is not writeable


### DATA ENRICHMENT (GEOCODING WITH GEOPY API)

In [5]:
import pyodbc
import pandas as pd
from geopy.geocoders import Nominatim
import time
import warnings
warnings.filterwarnings('ignore')

conn_str = "Driver={ODBC Driver 18 for SQL Server};Server=tcp:nashvilleserver2210.database.windows.net,1433;Database=NashvilleHousingDB;Uid=sqladmin;Pwd=Sr123456789@;Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
conn = pyodbc.connect(conn_str)

print("1. Extracting Address Data from your new Dim_Property table...")

query = """
SELECT TOP 5 
    PropertyAddress AS PropertySplitAddress, 
    PropertyCity AS PropertySplitCity
FROM dbo.Dim_Property
WHERE PropertyAddress IS NOT NULL;
"""

df = pd.read_sql(query, conn)

print("2. Setting up the Geocoder...")
geolocator = Nominatim(user_agent="nashville_portfolio_project")

df['Clean_Street'] = df['PropertySplitAddress'].str.strip().str.replace('  ', ' ', regex=False)
df['Clean_City'] = df['PropertySplitCity'].str.strip()

def get_smart_coordinates(row):
    full_address = f"{row['Clean_Street']}, {row['Clean_City']}, TN"
    city_address = f"{row['Clean_City']}, TN"
    
    try:
        
        location = geolocator.geocode(full_address, timeout=10)
        time.sleep(1)
        
        if location:
            return pd.Series([location.latitude, location.longitude, 'Exact Match'])
            
        
        location = geolocator.geocode(city_address, timeout=10)
        time.sleep(1)
        
        if location:
            return pd.Series([location.latitude, location.longitude, 'City Fallback'])
            
        return pd.Series([None, None, 'Not Found'])
        
    except Exception as e:
        return pd.Series([None, None, 'Error'])

print("3. Asking the internet for coordinates using smart fallback...")

df[['Latitude', 'Longitude', 'Match_Type']] = df.apply(get_smart_coordinates, axis=1)

print("\nDone! Here is your dynamically mapped data:")
print(df[['Clean_Street', 'Clean_City', 'Latitude', 'Longitude', 'Match_Type']])

1. Extracting Address Data from your new Dim_Property table...
2. Setting up the Geocoder...
3. Asking the internet for coordinates using smart fallback...

Done! Here is your dynamically mapped data:
        Clean_Street      Clean_City   Latitude  Longitude     Match_Type
0  1808 FOX CHASE DR  GOODLETTSVILLE  36.323107  -86.71333  City Fallback
1  1832 FOX CHASE DR  GOODLETTSVILLE  36.323107  -86.71333  City Fallback
2  1864 FOX CHASE DR  GOODLETTSVILLE  36.323107  -86.71333  City Fallback
3  1853 FOX CHASE DR  GOODLETTSVILLE  36.323107  -86.71333  City Fallback
4  1829 FOX CHASE DR  GOODLETTSVILLE  36.323107  -86.71333  City Fallback


### IMPORTS & DATABASE CONNECTION

In [6]:
import pyodbc
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

conn_str = "Driver={ODBC Driver 18 for SQL Server};Server=tcp:nashvilleserver2210.database.windows.net,1433;Database=NashvilleHousingDB;Uid=sqladmin;Pwd=Sr123456789@;Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
conn = pyodbc.connect(conn_str)

### DATA EXTRACTION

In [7]:
print("\n===== DATA EXTRACTION =====")

query = """
SELECT 
    f.SalePrice, 
    p.YearBuilt, 
    p.Bedrooms, 
    p.FullBath, 
    p.HalfBath, 
    p.Acreage
FROM dbo.Fact_Sales f
JOIN dbo.Dim_Property p ON f.PropertyKey = p.PropertyKey
WHERE f.SalePrice IS NOT NULL 
  AND p.YearBuilt IS NOT NULL 
  AND p.Bedrooms IS NOT NULL;
"""

df = pd.read_sql(query, conn)

print(f"Data Shape: {df.shape[0]} Rows, {df.shape[1]} Columns")
print(df.head().to_string())
print(df.describe().round(2).to_string())


===== DATA EXTRACTION =====
Data Shape: 25933 Rows, 6 Columns
   SalePrice  YearBuilt  Bedrooms  FullBath  HalfBath  Acreage
0   240000.0       1986         3       3.0       0.0      2.3
1   366000.0       1998         3       3.0       2.0      3.5
2   435000.0       1987         4       3.0       0.0      2.9
3   255000.0       1985         3       3.0       0.0      2.6
4   278000.0       1984         4       3.0       0.0      2.0
         SalePrice  YearBuilt  Bedrooms  FullBath  HalfBath   Acreage
count     25933.00   25933.00  25933.00  25932.00  25751.00  25933.00
mean     276517.94    1964.18      3.10      1.91      0.29      0.45
std      301914.63      27.09      0.85      0.95      0.49      0.77
min         100.00    1799.00      0.00      0.00      0.00      0.04
25%      125000.00    1948.00      3.00      1.00      0.00      0.18
50%      187500.00    1960.00      3.00      2.00      0.00      0.27
75%      326500.00    1983.00      4.00      2.00      1.00      0.45

### FEATURE ENGINEERING

In [8]:
print("\n===== FEATURE ENGINEERING =====")

df['PropertyAge'] = 2026 - df['YearBuilt']
df = df.drop(columns=['YearBuilt']).dropna()

print(df.head().to_string())


===== FEATURE ENGINEERING =====
   SalePrice  Bedrooms  FullBath  HalfBath  Acreage  PropertyAge
0   240000.0         3       3.0       0.0      2.3           40
1   366000.0         3       3.0       2.0      3.5           28
2   435000.0         4       3.0       0.0      2.9           39
3   255000.0         3       3.0       0.0      2.6           41
4   278000.0         4       3.0       0.0      2.0           42


### MODEL TRAINING

In [9]:
print("\n===== MODEL TRAINING =====")

X = df[['Bedrooms', 'FullBath', 'HalfBath', 'Acreage', 'PropertyAge']]
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"Training Rows: {X_train.shape[0]}")
print(f"Testing Rows: {X_test.shape[0]}")


===== MODEL TRAINING =====
Training Rows: 20600
Testing Rows: 5151


### MODEL EVALUATION

In [10]:
print("\n===== MODEL EVALUATION =====")

predictions = model.predict(X_test)

rmse = mean_squared_error(y_test, predictions) ** 0.5
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"RMSE : ${rmse:,.2f}")
print(f"MAE  : ${mae:,.2f}")
print(f"R2   : {r2:.4f}")

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df.to_string(index=False))


===== MODEL EVALUATION =====
RMSE : $218,541.21
MAE  : $119,697.95
R2   : 0.4680
    Feature  Importance
   FullBath    0.393114
    Acreage    0.264253
PropertyAge    0.224079
   HalfBath    0.065767
   Bedrooms    0.052787


## Export the Trained Model (Pickling)

In [11]:
import joblib

joblib.dump(model, 'nashville_rf_model.pkl')

features = X.columns.tolist()
joblib.dump(features, 'model_features.pkl')

print("Model successfully saved as 'nashville_rf_model.pkl'!")
print("Ready for Streamlit deployment!")

Model successfully saved as 'nashville_rf_model.pkl'!
Ready for Streamlit deployment!


## The Streamlit Blueprint

In [25]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

# --- 1. PAGE CONFIGURATION ---
st.set_page_config(
    page_title="Nashville Real Estate AVM",
    page_icon="🏙️",
    layout="wide"
)

# --- 2. CUSTOM CSS INJECTION ---
st.markdown("""
<style>
.stApp {
    background: linear-gradient(135deg, #0f172a, #1e293b);
    color: #f1f5f9;
}
footer {visibility: hidden;}
.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}
.card {
    background: white;
    padding: 25px;
    border-radius: 14px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.2);
    margin-bottom: 20px;
}
.price {
    font-size: 42px;
    font-weight: 700;
    color: #16a34a;
    margin-bottom: 0;
}
.stButton>button {
    width: 100%;
    height: 48px;
    border-radius: 10px;
    background-color: #2563eb;
    color: white;
    font-weight: 600;
    border: none;
    transition: 0.3s;
}
.stButton>button:hover {
    background-color: #1d4ed8;
}
.custom-footer {
    margin-top: 60px;
    padding: 30px;
    background: #0f172a;
    border-radius: 12px;
    text-align: center;
    font-size: 14px;
    color: #cbd5e1;
}
[data-testid="stMetricLabel"] {
    color: #475569 !important;
}
[data-testid="stMetricValue"] {
    color: #0f172a !important;
}
</style>
""", unsafe_allow_html=True)

# --- 3. LOAD THE ML ENGINE ---
@st.cache_resource
def load_model():
    model = joblib.load("nashville_rf_model.pkl")
    features = joblib.load("model_features.pkl")
    return model, features

model, features = load_model()
NASHVILLE_AVG = 450000

# --- 4. APP HEADER ---
st.title("🏙️ Nashville Automated Valuation Model")
st.caption("AI-powered real estate pricing engine using Davidson County housing data")
st.divider()

# --- 5. TWO-COLUMN LAYOUT ---
col1, col2 = st.columns([1,1])

with col1:
    st.subheader("🏡 Property Details")
    bedrooms = st.slider("Bedrooms", 1, 10, 3)
    full_bath = st.slider("Full Bathrooms", 1, 7, 2)
    half_bath = st.slider("Half Bathrooms", 0, 4, 0)
    acreage = st.number_input("Lot Size (Acres)", 0.1, 10.0, 0.5, step=0.1)
    age = st.slider("Property Age (Years)", 0, 150, 15)
    predict_button = st.button("Generate Valuation")

with col2:
    st.subheader("📊 Valuation Result")

    if predict_button:
        input_data = pd.DataFrame([[bedrooms, full_bath, half_bath, acreage, age]], columns=features)
        prediction = model.predict(input_data)[0]
        
        margin = prediction * 0.045
        low = prediction - margin
        high = prediction + margin
        diff = prediction - NASHVILLE_AVG
        diff_text = f"+${diff:,.0f} above avg" if diff > 0 else f"-${abs(diff):,.0f} below avg"

        st.markdown("<div class='card'>", unsafe_allow_html=True)
        st.markdown("<p style='color:#64748b;'>Estimated Market Value</p>", unsafe_allow_html=True)
        st.markdown(f"<p class='price'>${prediction:,.0f}</p>", unsafe_allow_html=True)
        st.markdown(f"<p style='color:#475569;'>Confidence Interval: ${low:,.0f} - ${high:,.0f}</p>", unsafe_allow_html=True)
        st.markdown("---")

        colA, colB = st.columns(2)
        with colA:
            st.metric("Vs Nashville Avg", diff_text)
        with colB:
            tier = "Premium / Luxury" if prediction > 750000 else "Mid-Market" if prediction > 350000 else "Starter Home"
            st.metric("Property Tier", tier)

        st.markdown("</div>", unsafe_allow_html=True)
        st.info(f"**AI Insight:** Property age ({age} yrs) and lot size ({acreage} acres) strongly influence this valuation.")
    else:
        st.markdown("<div class='card' style='text-align:center;'><p style='color:#94a3b8;'>Adjust inputs on the left and click <b>Generate Valuation</b> to see the AI's prediction.</p></div>", unsafe_allow_html=True)

# --- 6. POWER BI DASHBOARD SECTION (LIVE EMBED) ---
st.markdown("---")
st.subheader("📊 Market Analytics Dashboard")
st.markdown("<p style='color: #cbd5e1;'>Interact with the live Power BI dashboard below, connected directly to our Azure SQL database via DirectQuery.</p>", unsafe_allow_html=True)

# Your live Power BI URL
power_bi_url = "https://app.powerbi.com/view?r=eyJrIjoiZTM2ZGI3OGQtNjVjYS00ZTFlLWFmOTAtODUxZTg4MDU1ODhhIiwidCI6IjM0YmQ4YmVkLTJhYzEtNDFhZS05ZjA4LTRlMGEzZjExNzA2YyJ9"

# Embeds the live Power BI dashboard
st.markdown(f'<iframe title="Nashville Housing Dashboard" width="100%" height="650" src="{power_bi_url}" frameborder="0" allowFullScreen="true" style="border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.3);"></iframe>', unsafe_allow_html=True)

# --- 7. FOOTER ---
st.markdown("""
<div class='custom-footer'>
    <strong>Model & Dataset Information</strong><br><br>
    📊 Training Rows: ~20,600 | 📈 Testing Rows: ~5,150<br><br>
    Built with Python · Streamlit · Azure SQL · Scikit-Learn<br>
    Random Forest Regressor (R²: 0.468)<br><br>
    <em>Portfolio project. Not financial advice.</em>
</div>
""", unsafe_allow_html=True)

Overwriting app.py


## Create requirements.txt

In [1]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib

Writing requirements.txt


## Create .gitignore

In [2]:
%%writefile .gitignore
# Ignore data files to save space
data/
*.csv
*.xlsx

# Ignore system files
.ipynb_checkpoints/
__pycache__/
.DS_Store

Writing .gitignore
